# 13 · Candidate Generator Table

Builds `e4st_gen_table.parquet` — the combined table of existing generators
(build_status=`built`) and endogenous candidates (build_status=`unbuilt`) for E4ST.

**Cost data source:** NREL ATB 2024 v3.0.0, Moderate scenario, 2030 projection (hardcoded constants below).  
**No external API calls.** All inputs come from previously processed files.

In [1]:
import json
import warnings
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# ── paths ──────────────────────────────────────────────────────────────────────
ROOT   = Path('..').resolve()
PROC   = ROOT / 'data' / 'processed'

# ── NREL ATB 2024 cost constants (Moderate scenario, 2030 projection) ─────────
# Source: NREL ATB 2024 v3.0.0  https://atb.nrel.gov/electricity/2024/
ATB_2024 = {
    'wind_onshore':     {'capex_per_kw': 1_350, 'fom_per_kw_yr': 28,  'vom_per_mwh': 0.0,  'econ_life': 25},
    'wind_offshore':    {'capex_per_kw': 3_200, 'fom_per_kw_yr': 88,  'vom_per_mwh': 0.0,  'econ_life': 25},
    'solar_pv_utility': {'capex_per_kw':   900, 'fom_per_kw_yr': 17,  'vom_per_mwh': 0.0,  'econ_life': 30},
    'ng_cc':            {'capex_per_kw': 1_050, 'fom_per_kw_yr': 15,  'vom_per_mwh': 2.0,  'econ_life': 40},
    'ng_ct':            {'capex_per_kw':   700, 'fom_per_kw_yr': 10,  'vom_per_mwh': 3.0,  'econ_life': 30},
    'smr_nuclear':      {'capex_per_kw': 6_500, 'fom_per_kw_yr': 130, 'vom_per_mwh': 2.5,  'econ_life': 60},
    'battery_4h':       {'capex_per_kw':   650, 'fom_per_kw_yr': 10,  'vom_per_mwh': 0.5,  'econ_life': 20},
}

DISCOUNT_RATE    = 0.07
MODEL_HORIZON_YR = 30  # years over which capex is recovered in model (not used in annuity formula)

# ── technology pcap_max (MW) ───────────────────────────────────────────────────
PCAP_MAX = {
    'wind_onshore':     500.0,
    'solar_pv_utility': 400.0,
    'ng_cc':            600.0,
    'ng_ct':            300.0,
    'smr_nuclear':      300.0,
    'battery_4h':       200.0,
}

# ── regional BA lists ──────────────────────────────────────────────────────────
WECC_BAS = {
    'AZPS','AVA','AVRN','BANC','BPAT','CHPD','CISO','DEAA','DOPD',
    'EPE','GCPd','GCPD','GRIF','GRMA','GWA','HGMA','IID','IPCO',
    'LDWP','NEVP','NWMT','PACE','PACW','PGE','PNM','PSCO','PSEI',
    'SCL','SRP','TEPC','TIDC','TPWR','WACM','WALC','WAUW','WWA',
}
SERC_BAS = {
    'CPLE','CPLW','DUK','EKPC','EEI','FPC','FPL','GVL','HST',
    'JEA','LGEE','NSB','OVEC','SC','SCEG','SEC','SEPA','SOCO',
    'TAL','TEC','TVA','YAD',
}
SPP_BAS = {'SWPP', 'SPA', 'WAUW'}

# ── wind-resource BAs (get onshore wind candidates regardless of existing capacity) ─
WIND_RESOURCE_BAS = {'WACM', 'PACE', 'MISO', 'SWPP', 'PJM'}

# ── Wyoming geographic bounds (for PACE SMR filter) ───────────────────────────
WY_LAT_MIN, WY_LAT_MAX = 41.0, 45.0
WY_LON_MIN, WY_LON_MAX = -111.0, -104.0

# ── Idaho National Laboratory coordinates (SMR seed) ──────────────────────────
INL_LAT, INL_LON = 43.52, -112.65

# ── SMR nuclear 50-km proximity radius ────────────────────────────────────────
SMR_NUC_PROX_KM = 50.0

print('Constants loaded.')
print(f'  ATB technologies: {list(ATB_2024)}')
print(f'  Discount rate: {DISCOUNT_RATE:.0%}  |  Model horizon: {MODEL_HORIZON_YR} yr')

Constants loaded.
  ATB technologies: ['wind_onshore', 'wind_offshore', 'solar_pv_utility', 'ng_cc', 'ng_ct', 'smr_nuclear', 'battery_4h']
  Discount rate: 7%  |  Model horizon: 30 yr


In [2]:
# ── load data ─────────────────────────────────────────────────────────────────
buses   = gpd.read_file(PROC / 'synthetic_buses.geojson')
gens    = pd.read_parquet(PROC / 'generators_with_retirements.parquet')
spa     = pd.read_parquet(PROC / 'synthetic_plant_assignments.parquet')
av      = pd.read_parquet(PROC / 'e4st_availability_factors.parquet')
hours   = pd.read_csv(PROC / 'e4st_hours.csv')
meta    = json.loads((PROC / 'network_metadata.json').read_text())

# Attach synthetic bus_id (0-499) to generators via plant_assignments
gens_synth = gens.merge(
    spa[['plant_id', 'generator_id', 'bus_id']].rename(columns={'bus_id': 'synth_bus_id'}),
    on=['plant_id', 'generator_id'],
    how='left',
)
n_matched   = gens_synth['synth_bus_id'].notna().sum()
n_unmatched = gens_synth['synth_bus_id'].isna().sum()
print(f'Generators loaded:  {len(gens):,} total | {n_matched:,} matched to synthetic bus | {n_unmatched} unmatched (dropped)')
gens_synth = gens_synth[gens_synth['synth_bus_id'].notna()].copy()
gens_synth['synth_bus_id'] = gens_synth['synth_bus_id'].astype(int)

print(f'Synthetic buses:    {len(buses):,}')
print(f'Availability rows:  {len(av):,}  ({av["technology"].unique().tolist()})')
print(f'Representative hrs: {len(hours)}')

Generators loaded:  14,354 total | 14,353 matched to synthetic bus | 35 unmatched (dropped)
Synthetic buses:    500
Availability rows:  16,000  (['wind', 'solar'])
Representative hrs: 16


In [3]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorized haversine distance in km between (lat1,lon1) and (lat2,lon2) arrays."""
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi  = np.radians(lat2 - lat1)
    dlam  = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam / 2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


def annualized_capex_per_mw_per_hour(capex_per_kw, econ_life, discount_rate=DISCOUNT_RATE):
    """
    E4ST capex column: annualized capital cost expressed per MW per model-hour.

    Formula (capital recovery factor):
        capex_annual = capex_per_kw * 1000 * DR / (1 - (1 + DR)^(-life))
        capex_per_mw_per_hour = capex_annual / 8760

    where capex_per_kw * 1000 converts $/kW → $/MW.
    """
    capex_per_mw  = capex_per_kw * 1_000
    crf           = discount_rate / (1 - (1 + discount_rate) ** (-econ_life))
    capex_annual  = capex_per_mw * crf
    return capex_annual / 8_760


# Pre-compute capex for all ATB technologies
atb_capex = {}
for tech, v in ATB_2024.items():
    c = annualized_capex_per_mw_per_hour(v['capex_per_kw'], v['econ_life'])
    atb_capex[tech] = c
    print(f'  {tech:<20} capex_per_kw={v["capex_per_kw"]:>6,}  life={v["econ_life"]:>2}yr  → capex={c:8.2f} $/MWh-hr')

  wind_onshore         capex_per_kw= 1,350  life=25yr  → capex=   13.22 $/MWh-hr
  wind_offshore        capex_per_kw= 3,200  life=25yr  → capex=   31.35 $/MWh-hr
  solar_pv_utility     capex_per_kw=   900  life=30yr  → capex=    8.28 $/MWh-hr
  ng_cc                capex_per_kw= 1,050  life=40yr  → capex=    8.99 $/MWh-hr
  ng_ct                capex_per_kw=   700  life=30yr  → capex=    6.44 $/MWh-hr
  smr_nuclear          capex_per_kw= 6,500  life=60yr  → capex=   52.85 $/MWh-hr
  battery_4h           capex_per_kw=   650  life=20yr  → capex=    7.00 $/MWh-hr


In [4]:
# ── weighted mean CF per bus per technology ────────────────────────────────────
# Hours are weighted by fractional year share; use those weights for the mean.
av_w = av.merge(hours[['hour_id', 'weight']], on='hour_id')
mean_cf = (
    av_w.groupby(['bus_id', 'technology'])
    .apply(lambda g: np.average(g['cf'], weights=g['weight']), include_groups=False)
    .reset_index(name='mean_cf')
)

wind_cf  = mean_cf[mean_cf['technology'] == 'wind'].set_index('bus_id')['mean_cf']
solar_cf = mean_cf[mean_cf['technology'] == 'solar'].set_index('bus_id')['mean_cf']

print(f'Wind  CF — mean={wind_cf.mean():.3f}  p5={wind_cf.quantile(.05):.3f}  p95={wind_cf.quantile(.95):.3f}')
print(f'Solar CF — mean={solar_cf.mean():.3f}  p5={solar_cf.quantile(.05):.3f}  p95={solar_cf.quantile(.95):.3f}')

Wind  CF — mean=0.277  p5=0.140  p95=0.426
Solar CF — mean=0.166  p5=0.137  p95=0.207


In [5]:
# ── Seed bus sets for each candidate technology ────────────────────────────────

bus_ba    = buses.set_index('bus_id')['ba_code']
bus_lat   = buses.set_index('bus_id')['lat']
bus_lon   = buses.set_index('bus_id')['lon']
all_buses = set(buses['bus_id'].values)

# 1. Wind onshore — buses with existing wind OR in WIND_RESOURCE_BAS
wind_existing_buses = set(
    gens_synth[gens_synth['technology'].isin(['Onshore Wind Turbine', 'Offshore Wind Turbine'])]
    ['synth_bus_id'].unique()
)
wind_resource_buses = set(buses[buses['ba_code'].isin(WIND_RESOURCE_BAS)]['bus_id'].values)
wind_seed_buses     = wind_existing_buses | wind_resource_buses
print(f'Wind seed buses: {len(wind_seed_buses)}  (existing wind: {len(wind_existing_buses)}, resource BA: {len(wind_resource_buses)})')

# 2. Solar — all buses in WECC ∪ SERC ∪ SPP
solar_ba_set    = WECC_BAS | SERC_BAS | SPP_BAS
solar_seed_buses = set(buses[buses['ba_code'].isin(solar_ba_set)]['bus_id'].values)
print(f'Solar seed buses: {len(solar_seed_buses)}  (WECC∪SERC∪SPP)')

# 3. NG CC / NG CT / Battery — all buses
print(f'NG CC / NG CT / Battery seed buses: {len(all_buses)}  (all buses)')

# 4. SMR nuclear
#    a. All WACM buses
smr_wacm = set(buses[buses['ba_code'] == 'WACM']['bus_id'].values)
#    b. PACE buses inside Wyoming geographic bounds (lat 41-45, lon -111 to -104)
pace_buses_df = buses[buses['ba_code'] == 'PACE'].copy()
pace_wy       = pace_buses_df[
    (pace_buses_df['lat'] >= WY_LAT_MIN) & (pace_buses_df['lat'] <= WY_LAT_MAX) &
    (pace_buses_df['lon'] >= WY_LON_MIN) & (pace_buses_df['lon'] <= WY_LON_MAX)
]
smr_pace_wy   = set(pace_wy['bus_id'].values)
print(f'\nPACE buses total: {len(pace_buses_df)}')
print(f'PACE buses passing Wyoming geographic filter: {len(smr_pace_wy)}')
print(pace_wy[['bus_id','lat','lon']].to_string())

#    c. INL bus (nearest synthetic bus to lat=43.52, lon=-112.65)
bus_arr  = buses[['lat','lon']].values
inl_dists = haversine_km(bus_arr[:,0], bus_arr[:,1], INL_LAT, INL_LON)
inl_bus_id = int(buses.iloc[np.argmin(inl_dists)]['bus_id'])
inl_ba     = bus_ba[inl_bus_id]
print(f'\nINL nearest bus: id={inl_bus_id}  ba={inl_ba}  dist={inl_dists.min():.1f} km')

#    d. Buses within 50 km of any existing nuclear plant bus
nuc_synth_ids = gens_synth[gens_synth['technology'] == 'Nuclear']['synth_bus_id'].unique()
nuc_lats = buses[buses['bus_id'].isin(nuc_synth_ids)]['lat'].values
nuc_lons = buses[buses['bus_id'].isin(nuc_synth_ids)]['lon'].values

smr_prox_buses = set()
for blat, blon in zip(nuc_lats, nuc_lons):
    d = haversine_km(bus_arr[:,0], bus_arr[:,1], blat, blon)
    smr_prox_buses.update(buses[d <= SMR_NUC_PROX_KM]['bus_id'].values)
print(f'Buses within {SMR_NUC_PROX_KM:.0f} km of existing nuclear: {len(smr_prox_buses)}')

smr_seed_buses = smr_wacm | smr_pace_wy | {inl_bus_id} | smr_prox_buses
print(f'\nSMR total seed buses: {len(smr_seed_buses)}')
smr_wy_buses = smr_wacm | smr_pace_wy | ({inl_bus_id} if inl_ba in {'WACM','PACE','IPCO'} else set())
print(f'Wyoming-specific SMR buses (WACM + PACE-WY + INL): {len(smr_wacm | smr_pace_wy | {inl_bus_id})}')

Wind seed buses: 274  (existing wind: 151, resource BA: 202)
Solar seed buses: 254  (WECC∪SERC∪SPP)
NG CC / NG CT / Battery seed buses: 500  (all buses)

PACE buses total: 12
PACE buses passing Wyoming geographic filter: 6
     bus_id        lat         lon
282     282  41.722667 -108.868877
285     285  42.969420 -105.914698
286     286  41.940110 -110.890435
288     288  41.685853 -106.705102
291     291  44.317281 -109.696986
292     292  44.372718 -106.076090

INL nearest bus: id=155  ba=IPCO  dist=69.9 km
Buses within 50 km of existing nuclear: 67

SMR total seed buses: 86
Wyoming-specific SMR buses (WACM + PACE-WY + INL): 19


In [6]:
# ── Build candidate rows ───────────────────────────────────────────────────────

def make_candidates(bus_ids, tech_key, genfuel, gentype, emis_co2_rate):
    """Return a DataFrame of candidate generator rows for the given bus set."""
    v   = ATB_2024[tech_key]
    cpx = atb_capex[tech_key]
    rows = []
    for bid in sorted(bus_ids):
        rows.append({
            'bus_id':       bid,
            'build_status': 'unbuilt',
            'build_type':   'endog',
            'year_on':      2025,
            'year_shutdown': pd.NA,
            'econ_life':    v['econ_life'],
            'genfuel':      genfuel,
            'gentype':      gentype,
            'pcap0':        0.0,
            'pcap_min':     0.0,
            'pcap_max':     PCAP_MAX[tech_key],
            'vom':          v['vom_per_mwh'],
            'fom':          v['fom_per_kw_yr'],
            'capex':        cpx,
            'heat_rate':    0.0,
            'fuel_price':   0.0,
            'cf_hist':      0.0,
            'emis_co2_rate': emis_co2_rate,
            'plant_id':     pd.NA,
            'generator_id': pd.NA,
            'stateid':      pd.NA,
        })
    return pd.DataFrame(rows)


cand_wind    = make_candidates(wind_seed_buses,  'wind_onshore',     'wind',    'wind_onshore',     0.0)
cand_solar   = make_candidates(solar_seed_buses, 'solar_pv_utility', 'solar',   'solar_pv',         0.0)
cand_ng_cc   = make_candidates(all_buses,        'ng_cc',            'ng',      'ng_cc',            0.40)
cand_ng_ct   = make_candidates(all_buses,        'ng_ct',            'ng',      'ng_ct',            0.55)
cand_smr     = make_candidates(smr_seed_buses,   'smr_nuclear',      'nuclear', 'smr',              0.0)
cand_battery = make_candidates(all_buses,        'battery_4h',       'elec',    'battery_4h',       0.0)

print(f'Candidate rows (pre CF filter):')
for name, df in [('wind', cand_wind), ('solar', cand_solar), ('ng_cc', cand_ng_cc),
                 ('ng_ct', cand_ng_ct), ('smr', cand_smr), ('battery', cand_battery)]:
    print(f'  {name:<12} {len(df):>5} rows')

Candidate rows (pre CF filter):
  wind           274 rows
  solar          254 rows
  ng_cc          500 rows
  ng_ct          500 rows
  smr             86 rows
  battery        500 rows


In [7]:
# ── CF join and low-resource filter ───────────────────────────────────────────

CF_FLOOR_WIND = 0.15  # pcap_max → 0 if mean wind CF below this threshold

# Wind: join mean CF, flag and zero out low-resource buses
cand_wind = cand_wind.merge(
    wind_cf.rename('mean_cf').reset_index(),
    on='bus_id', how='left'
)
cand_wind['mean_cf'] = cand_wind['mean_cf'].fillna(0.0)
cand_wind['cf_hist'] = cand_wind['mean_cf']

low_wind_mask = cand_wind['mean_cf'] < CF_FLOOR_WIND
n_low_wind    = low_wind_mask.sum()
n_zero_cf     = (cand_wind['mean_cf'] == 0.0).sum()
cand_wind.loc[low_wind_mask, 'pcap_max'] = 0.0

print(f'Wind candidates with mean CF < {CF_FLOOR_WIND}: {n_low_wind} excluded (pcap_max set to 0)')
print(f'  of which CF == 0.0: {n_zero_cf}')

# Confirm 4 Alaska/Canada zero-CF buses appear in excluded set
zero_cf_bus_ids = cand_wind[cand_wind['mean_cf'] == 0.0]['bus_id'].tolist()
zero_cf_bas     = buses[buses['bus_id'].isin(zero_cf_bus_ids)][['bus_id','ba_code']]
print(f'\nZero-CF wind buses (Alaska + out-of-CONUS):')
print(zero_cf_bas.to_string(index=False))

# Solar: join mean CF
cand_solar = cand_solar.merge(
    solar_cf.rename('mean_cf').reset_index(),
    on='bus_id', how='left'
)
cand_solar['mean_cf'] = cand_solar['mean_cf'].fillna(0.0)
cand_solar['cf_hist'] = cand_solar['mean_cf']

# Confirm Wyoming wind CF (WACM buses)
wacm_bus_ids = buses[buses['ba_code'] == 'WACM']['bus_id'].values
pace_wy_ids  = list(smr_pace_wy)  # PACE WY-filtered buses
wy_wind_cands = cand_wind[cand_wind['bus_id'].isin(list(wacm_bus_ids) + pace_wy_ids)]
print(f'\nWyoming wind candidates mean CF: {wy_wind_cands["mean_cf"].mean():.3f}  (threshold ≥0.30)')
assert wy_wind_cands['mean_cf'].mean() >= 0.30, 'Wyoming wind CF below 0.30!'
print('  Wyoming wind CF check PASSED')

# Drop the mean_cf working column from both (cf_hist already set)
cand_wind  = cand_wind.drop(columns=['mean_cf'])
cand_solar = cand_solar.drop(columns=['mean_cf'])

Wind candidates with mean CF < 0.15: 7 excluded (pcap_max set to 0)
  of which CF == 0.0: 0

Zero-CF wind buses (Alaska + out-of-CONUS):
Empty DataFrame
Columns: [bus_id, ba_code]
Index: []

Wyoming wind candidates mean CF: 0.336  (threshold ≥0.30)
  Wyoming wind CF check PASSED


In [8]:
# ── Map existing generators to output schema ───────────────────────────────────

# genfuel: E4ST-style fuel label derived from EIA fuel_type
FUELTYPE_TO_GENFUEL = {
    'Natural Gas': 'ng',
    'Gaseous Propane': 'ng',
    'Nuclear': 'nuclear',
    'Water': 'hydro',
    'Wind': 'wind',
    'Solar': 'solar',
    'Geothermal': 'geo',
    'Electricity used for energy storage': 'elec',
    'Bituminous Coal': 'coal',
    'Subbituminous Coal': 'coal',
    'Lignite': 'coal',
    'Refined Coal': 'coal',
    'Coal-Derived Synthesis Gas': 'coal',
    'Waste Coal': 'coal',
    'Residual Fuel Oil': 'oil',
    'Disillate Fuel Oil': 'oil',
    'Jet Fuel': 'oil',
    'Kerosene': 'oil',
    'Petroleum Coke': 'petcoke',
    'Landfill Gas': 'gas_other',
    'Other Gas': 'gas_other',
    'Blast-Furnace Gas': 'gas_other',
    'Other Biomass Gases ': 'biomass',
    'Other Biomass Liquids ': 'biomass',
    'Wood Waste Solids': 'biomass',
    'Wood Waste Liquids': 'biomass',
    'Agriculture Byproducts': 'biomass',
    'Black Liquor': 'biomass',
    'Municipal Solid Waste (All)': 'waste',
    'Other': 'other',
    'Purchased Steam': 'other',
    'Waste Heat': 'other',
}

# gentype: E4ST-style plant type label derived from EIA technology
TECHNOLOGY_TO_GENTYPE = {
    'Natural Gas Fired Combined Cycle': 'ng_cc',
    'Natural Gas Fired Combustion Turbine': 'ng_ct',
    'Natural Gas Steam Turbine': 'ng_steam',
    'Natural Gas Internal Combustion Engine': 'ng_ice',
    'Natural Gas with Compressed Air Storage': 'ng_caes',
    'Other Natural Gas': 'ng_other',
    'Conventional Steam Coal': 'coal_steam',
    'Coal Integrated Gasification Combined Cycle': 'coal_igcc',
    'Nuclear': 'nuclear',
    'Conventional Hydroelectric': 'hydro',
    'Hydroelectric Pumped Storage': 'hydro_pumped',
    'Onshore Wind Turbine': 'wind_onshore',
    'Offshore Wind Turbine': 'wind_offshore',
    'Solar Photovoltaic': 'solar_pv',
    'Solar Thermal with Energy Storage': 'solar_thermal',
    'Solar Thermal without Energy Storage': 'solar_thermal',
    'Batteries': 'battery',
    'Flywheels': 'flywheel',
    'Geothermal': 'geothermal',
    'Landfill Gas': 'landfill_gas',
    'Municipal Solid Waste': 'msw',
    'Wood/Wood Waste Biomass': 'biomass',
    'Other Waste Biomass': 'biomass',
    'Petroleum Liquids': 'oil_ct',
    'Petroleum Coke': 'petcoke_boiler',
    'Other Gases': 'other_gas',
    'All Other': 'other',
}

def emis_rate_for_existing(genfuel, gentype):
    """Return tCO2/MWh for existing generators based on genfuel/gentype."""
    if genfuel == 'coal':    return 0.98
    if genfuel == 'ng':
        # CC is more efficient; CT / steam / ICE use higher rate
        return 0.40 if gentype in ('ng_cc', 'ng_caes') else 0.55
    if genfuel == 'oil':     return 0.65
    if genfuel == 'petcoke': return 0.95
    if genfuel in ('nuclear', 'hydro', 'wind', 'solar', 'geo', 'elec'): return 0.0
    return 0.0  # biomass, waste, other treated as 0 (biogenic)


e = gens_synth.copy()
e['genfuel']  = e['fuel_type'].map(FUELTYPE_TO_GENFUEL).fillna('other')
e['gentype']  = e['technology'].map(TECHNOLOGY_TO_GENTYPE).fillna('other')
e['emis_co2_rate'] = e.apply(lambda r: emis_rate_for_existing(r['genfuel'], r['gentype']), axis=1)

existing = pd.DataFrame({
    'bus_id':        e['synth_bus_id'],
    'build_status':  'built',
    'build_type':    'real',
    'year_on':       e['year_on'],
    'year_shutdown': e['year_shutdown'],
    'econ_life':     e['econ_life'],
    'genfuel':       e['genfuel'],
    'gentype':       e['gentype'],
    'pcap0':         e['capacity_mw'],
    'pcap_min':      0.0,
    'pcap_max':      e['capacity_mw'],
    'vom':           e['vom_per_mwh'],
    'fom':           e['fom_per_kw_yr'],
    'capex':         0.0,  # sunk cost — not annualized in dispatch model
    'heat_rate':     e['heat_rate_mmbtu_mwh'].fillna(0.0),
    'fuel_price':    e['fuel_cost_per_mmbtu'].fillna(0.0),
    'cf_hist':       0.0,
    'emis_co2_rate': e['emis_co2_rate'],
    'plant_id':      e['plant_id'],
    'generator_id':  e['generator_id'],
    'stateid':       e['stateid'],
})

print(f'Existing generator rows: {len(existing):,}')
print(f'genfuel distribution:')
print(existing['genfuel'].value_counts().to_string())

Existing generator rows: 14,353
genfuel distribution:
genfuel
ng           5057
hydro        3671
oil          1620
gas_other    1050
solar         930
wind          874
coal          424
biomass       317
geo           124
nuclear        94
waste          70
other          58
elec           46
petcoke        18


In [9]:
# ── Concatenate all rows and enforce column order ──────────────────────────────

COLS = [
    'bus_id', 'build_status', 'build_type', 'year_on', 'year_shutdown',
    'econ_life', 'genfuel', 'gentype', 'pcap0', 'pcap_min', 'pcap_max',
    'vom', 'fom', 'capex', 'heat_rate', 'fuel_price', 'cf_hist',
    'emis_co2_rate', 'plant_id', 'generator_id', 'stateid',
]

candidates = pd.concat(
    [cand_wind, cand_solar, cand_ng_cc, cand_ng_ct, cand_smr, cand_battery],
    ignore_index=True,
)[COLS]

gen_table = pd.concat([existing[COLS], candidates], ignore_index=True)[COLS]

# Cast types
gen_table['bus_id']   = gen_table['bus_id'].astype(int)
gen_table['year_on']  = pd.array(gen_table['year_on'],  dtype='Int64')
gen_table['year_shutdown'] = pd.array(gen_table['year_shutdown'], dtype='Int64')

out_path = PROC / 'e4st_gen_table.parquet'
gen_table.to_parquet(out_path, index=False)
print(f'Saved {out_path}  shape={gen_table.shape}')
print(f'Columns: {gen_table.columns.tolist()}')

Saved /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/e4st_gen_table.parquet  shape=(16467, 21)
Columns: ['bus_id', 'build_status', 'build_type', 'year_on', 'year_shutdown', 'econ_life', 'genfuel', 'gentype', 'pcap0', 'pcap_min', 'pcap_max', 'vom', 'fom', 'capex', 'heat_rate', 'fuel_price', 'cf_hist', 'emis_co2_rate', 'plant_id', 'generator_id', 'stateid']


In [10]:
# ── Summary statistics ─────────────────────────────────────────────────────────

n_total    = len(gen_table)
n_existing = (gen_table['build_status'] == 'built').sum()
n_cand     = (gen_table['build_status'] == 'unbuilt').sum()

print(f'Total rows:       {n_total:>8,}')
print(f'  Existing built: {n_existing:>8,}')
print(f'  Candidates:     {n_cand:>8,}')
print()

# Candidate counts and total MW by technology
cand_df = gen_table[gen_table['build_status'] == 'unbuilt']
cand_summary = (
    cand_df.groupby('gentype')
    .agg(n_candidates=('bus_id','count'), total_mw=('pcap_max','sum'))
    .reset_index()
    .sort_values('total_mw', ascending=False)
)
print('Candidate MW by technology:')
print(cand_summary.to_string(index=False))
print()

# Wyoming WACM and PACE separately
wacm_ids = set(buses[buses['ba_code']=='WACM']['bus_id'].values)
pace_ids = set(buses[buses['ba_code']=='PACE']['bus_id'].values)

wacm_cand = cand_df[cand_df['bus_id'].isin(wacm_ids)]
pace_cand = cand_df[cand_df['bus_id'].isin(pace_ids)]

print('Wyoming (WACM) candidate MW by technology:')
print(wacm_cand.groupby('gentype')['pcap_max'].sum().sort_values(ascending=False).to_string())
print()
print('Wyoming (PACE) candidate MW by technology:')
print(pace_cand.groupby('gentype')['pcap_max'].sum().sort_values(ascending=False).to_string())

# SMR Wyoming count
smr_wyoming_bus_ids = set(buses[
    (buses['ba_code'].isin(['WACM','PACE'])) &
    (
        (buses['ba_code'] == 'WACM') |
        (
            (buses['lat'] >= WY_LAT_MIN) & (buses['lat'] <= WY_LAT_MAX) &
            (buses['lon'] >= WY_LON_MIN) & (buses['lon'] <= WY_LON_MAX)
        )
    )
]['bus_id'].values) | {inl_bus_id}
n_smr_wy = cand_df[(cand_df['gentype']=='smr') & (cand_df['bus_id'].isin(smr_wyoming_bus_ids))].shape[0]
print(f'\nWyoming SMR candidate buses (WACM + PACE-WY + INL): {n_smr_wy}')

# Save summary CSV
cand_summary.to_csv(PROC / 'candidate_summary.csv', index=False)
print(f'\nSaved candidate_summary.csv')

Total rows:         16,467
  Existing built:   14,353
  Candidates:        2,114

Candidate MW by technology:
     gentype  n_candidates  total_mw
       ng_cc           500  300000.0
       ng_ct           500  150000.0
wind_onshore           274  133500.0
    solar_pv           254  101600.0
  battery_4h           500  100000.0
         smr            86   25800.0

Wyoming (WACM) candidate MW by technology:
gentype
ng_cc           7200.0
wind_onshore    6000.0
solar_pv        4800.0
ng_ct           3600.0
smr             3600.0
battery_4h      2400.0

Wyoming (PACE) candidate MW by technology:
gentype
ng_cc           7200.0
wind_onshore    6000.0
solar_pv        4800.0
ng_ct           3600.0
battery_4h      2400.0
smr             1800.0

Wyoming SMR candidate buses (WACM + PACE-WY + INL): 19

Saved candidate_summary.csv


In [11]:
# ── Update network_metadata.json ───────────────────────────────────────────────

meta['candidate_generators'] = {
    'n_candidates':   int(n_cand),
    'n_existing':     int(n_existing),
    'n_smr_wyoming':  int(n_smr_wy),
    'timestamp':      datetime.now(timezone.utc).isoformat(),
}

(PROC / 'network_metadata.json').write_text(json.dumps(meta, indent=4))
print('network_metadata.json updated')
print(json.dumps(meta['candidate_generators'], indent=2))

network_metadata.json updated
{
  "n_candidates": 2114,
  "n_existing": 14353,
  "n_smr_wyoming": 19,
  "timestamp": "2026-05-07T00:11:46.122839+00:00"
}


In [12]:
# ── Handoff confirmation ───────────────────────────────────────────────────────

errors = []

# 1. File exists
if not (PROC / 'e4st_gen_table.parquet').exists():
    errors.append('e4st_gen_table.parquet not found')
else:
    print('e4st_gen_table.parquet exists            OK')

# 2. Both build_status values present
statuses = set(gen_table['build_status'].unique())
if statuses == {'built', 'unbuilt'}:
    print(f'build_status values: {sorted(statuses)}   OK')
else:
    errors.append(f'Expected {{built, unbuilt}}, got {statuses}')

# 3. Wyoming SMR count > 0
if n_smr_wy > 0:
    print(f'Wyoming SMR candidates: {n_smr_wy}               OK')
else:
    errors.append('No Wyoming SMR candidates found')

# 4. No nulls in critical columns
critical_cols = ['vom', 'fom', 'capex', 'pcap0', 'pcap_max']
null_counts   = gen_table[critical_cols].isnull().sum()
if null_counts.sum() == 0:
    print(f'No nulls in {critical_cols}   OK')
else:
    errors.append(f'Nulls found:\n{null_counts[null_counts>0]}')

# 5. CF filter — confirm 4 zero-CF buses excluded (pcap_max=0)
n_zero_pcap = cand_wind[cand_wind['pcap_max'] == 0.0].shape[0]
print(f'Wind candidates with pcap_max=0: {n_zero_pcap} (low-resource excluded)')

# 6. Column schema
missing_cols = [c for c in COLS if c not in gen_table.columns]
if not missing_cols:
    print(f'All {len(COLS)} required columns present   OK')
else:
    errors.append(f'Missing columns: {missing_cols}')

print()
if errors:
    for e in errors:
        print(f'FAIL: {e}')
    raise RuntimeError('Handoff checks failed — see above')
else:
    print('HANDOFF CONDITION MET')

e4st_gen_table.parquet exists            OK
build_status values: ['built', 'unbuilt']   OK
Wyoming SMR candidates: 19               OK
No nulls in ['vom', 'fom', 'capex', 'pcap0', 'pcap_max']   OK
Wind candidates with pcap_max=0: 7 (low-resource excluded)
All 21 required columns present   OK

HANDOFF CONDITION MET
